# CSC 786 Class Activity Module 5 Activity 1 - Gunnar Andrews


# Ethical Reminder

Before collecting any data:

- Check Terms of Service and rate limits.  
- Avoid collecting or storing personally identifiable information (PII).  
- Document every endpoint, parameter, and date of collection.  
- Keep secrets (API keys) out of public repositories.  


### Setup Environment

In [ ]:
!pip install python-dotenv --quiet
import os, pandas as pd, requests, hashlib, json, sys, time
from datetime import datetime, timezone
from pathlib import Path

if not os.path.exists("/content/csc786-ethics-gunnar"):
    !git clone https://github.com/g0ldencybersec/csc786-ethics-gunnar.git /content/csc786-ethics-gunnar
%cd /content/csc786-ethics-gunnar

ROOT = Path("/content/csc786-ethics-gunnar")
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
print("Environment ready. Files will be stored in:", DATA)


### Enter OpenWeatherMap API Key

In [ ]:
from getpass import getpass
# OpenWeatherMap (requires free key)
# Register: https://home.openweathermap.org/users/sign_up

# You will be prompted for your Openweather API key here as to not save a key in version control

API_KEY = getpass("OpenWeatherMap API Key: ")
print("Key loaded:", API_KEY[:6] + "****" if API_KEY else "No key found")


### Fetch Data Using OpenWeather API

In [ ]:
url = "https://api.openweathermap.org/data/2.5/weather"
params = {"q": "Sioux Falls", "appid": API_KEY, "units": "metric"}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()
data = r.json()

weather = {
    "city": data["name"],
    "temperature": data["main"]["temp"],
    "humidity": data["main"]["humidity"],
    "condition": data["weather"][0]["description"]
}
weather


In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
out_json = DATA / f"current_weather_{timestamp}.json"
df.to_csv(out_csv, index=False)

file_hash = hashlib.sha256(out_csv.read_bytes()).hexdigest()

meta = {
    "timestamp_utc": timestamp,
    "endpoint": ENDPOINT,
    "params": PARAMS,
    "output": out_csv.name,
    "sha256": file_hash,
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "requests": requests.__version__,
}

with open(ROOT / "DATA_README.md", "a") as f:
    f.write(f"\n- {json.dumps(meta)}")

print(f"Saved {out_csv.name}, hash={file_hash[:10]}…")
!tail -n 3 DATA_README.md


### Public API Example Using Open Meteo

In [ ]:
ENDPOINT = "https://api.open-meteo.com/v1/forecast"
PARAMS = {
    "latitude": 44.6,
    "longitude": -96.8,
    "hourly": "temperature_2m",
    "forecast_days": 1
}

for attempt in range(3):
    try:
        r = requests.get(ENDPOINT, params=PARAMS, timeout=10)
        r.raise_for_status()
        break
    except requests.exceptions.RequestException as e:
        wait = 2 ** attempt
        print(f"Retrying in {wait}s due to: {e}")
        time.sleep(wait)

data = r.json()

df = pd.DataFrame({
    "time": data["hourly"]["time"],
    "temperature_2m": data["hourly"]["temperature_2m"]
})
df.head()


### Save Data and Log Provenance

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
out_csv = DATA / f"hourly_temps_{timestamp}.csv"
df.to_csv(out_csv, index=False)

file_hash = hashlib.sha256(out_csv.read_bytes()).hexdigest()

meta = {
    "timestamp_utc": timestamp,
    "endpoint": ENDPOINT,
    "params": PARAMS,
    "output": out_csv.name,
    "sha256": file_hash,
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "requests": requests.__version__,
}

with open(ROOT / "DATA_README.md", "a") as f:
    f.write(f"\n- {json.dumps(meta)}")

print(f"Saved {out_csv.name}, hash={file_hash[:10]}…")
!tail -n 3 DATA_README.md


### Verify everything worked

In [ ]:
!ls -lh /content
!ls -lh /content/data
!tail -n 5 DATA_README.md